In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%pip install evaluate transformers datasets rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=5eaace664ab3cd0eb2951e51ae80eca2203c3ff9903571f0c282d8eb7a0b10bb
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
import numpy as np
import evaluate
import torch

# 1. Đọc dataset từ CSV (file trong Drive)
# CSV phải có 2 cột: "article" và "summary"
df = pd.read_csv("/content/drive/MyDrive/dataset_all NLP.csv")

# 2. Chia dữ liệu: 80% train, 10% validation, 10% test
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test": Dataset.from_pandas(test_df),
})

# 3. Tokenizer
model_name = "vinai/bartpho-syllable"
tokenizer = AutoTokenizer.from_pretrained(model_name)

max_input_length = 512
max_target_length = 128

def preprocess_function(examples):
    inputs = examples["article"]
    targets = examples["summary"]

    model_inputs = tokenizer(
        inputs,
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=max_target_length,
            truncation=True,
            padding="max_length"
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 4. Tokenize dataset
tokenized_datasets = dataset.map(preprocess_function, batched=True)

# 5. Load model
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 6. Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 7. Định nghĩa tham số huấn luyện
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/bartpho-results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=12,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_dir="/content/drive/MyDrive/logs_bartpho",
    logging_steps=50,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    load_best_model_at_end=True,
)

# 8. Metric (ROUGE)
metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = ["\n".join(pred.strip().split(".")) for pred in decoded_preds]
    decoded_labels = ["\n".join(label.strip().split(".")) for label in decoded_labels]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)

    formatted_result = {}
    for key, value in result.items():
        if isinstance(value, (float, np.floating)):
            formatted_result[key] = value * 100
        else:
            formatted_result[key] = value.mid.fmeasure * 100

    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in predictions]
    formatted_result["gen_len"] = np.mean(prediction_lens)

    return formatted_result

# 9. Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 10. Train & lưu model
trainer.train()
trainer.save_model("/content/drive/MyDrive/bartpho-summary-model")

Map:   0%|          | 0/3041 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:4007: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/380 [00:00<?, ? examples/s]

Map:   0%|          | 0/381 [00:00<?, ? examples/s]

/tmp/ipython-input-356330257.py:115: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


...

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dtc225210147 (dtc225210147-samsung-electronics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum,Gen Len
1,0.282900,0.191025,57.644373,49.166967,54.084474,54.541361,21.000000
2,0.220400,0.187189,57.563795,49.133742,54.009486,54.458453,21.000000
3,0.130600,0.194698,57.589739,49.145978,53.786665,54.329814,21.000000
4,0.133800,0.207491,57.233737,49.001927,53.574577,54.049185,21.000000
5,0.089800,0.231635,57.507502,49.379301,53.951535,54.421372,21.000000
6,0.066900,0.250591,57.805379,49.402404,54.152381,54.640125,21.000000
7,0.046400,0.266643,57.726126,49.614269,53.956841,54.455301,21.000000
8,0.033800,0.287196,57.801593,49.503533,54.160658,54.618676,21.000000
9,0.026400,0.295100,57.795352,49.257372,54.090846,54.442660,21.000000
10,0.021900,0.300720,57.625129,49.589704,54.034031,54.454511,21.000000


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


In [ ]:
# Đánh giá mô hình trên tập validation
results = trainer.evaluate()

print("\n===== 📊 KẾT QUẢ ĐÁNH GIÁ ROUGE =====")
for key, value in results.items():
    if key.startswith("eval_rouge"):
        print(f"{key:<15}: {value:.2f}")



===== 📊 KẾT QUẢ ĐÁNH GIÁ ROUGE =====
eval_rouge1    : 57.84
eval_rouge2    : 49.91
eval_rougeL    : 54.31
eval_rougeLsum : 54.74


In [ ]:
import gradio as gr
import requests
from bs4 import BeautifulSoup
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Load model và tokenizer (đường dẫn model đã lưu trên Drive)
model_path = "/content/drive/MyDrive/bartpho-summary-model"  # đổi path của bạn
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# 2. Hàm lấy nội dung từ link bài báo
def get_article_text(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        # Lấy text trong <p>
        paragraphs = [p.get_text() for p in soup.find_all("p")]
        article_text = " ".join(paragraphs)
        return article_text
    except Exception as e:
        return f"Lỗi khi tải bài báo: {e}"

# 3. Hàm sinh tóm tắt
def summarize_from_url(url, max_input_len=512, max_output_len=128):
    article = get_article_text(url)
    if article.startswith("Lỗi"):
        return article  # báo lỗi luôn nếu không tải được

    inputs = tokenizer(
        article,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_len,
    )
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=max_output_len,
        num_beams=4,
        early_stopping=True
    )
    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

# 4. Tạo Gradio UI
demo = gr.Interface(
    fn=summarize_from_url,
    inputs=gr.Textbox(label="Nhập link bài báo"),
    outputs=gr.Textbox(label="Tóm tắt"),
    title="BARTPho - Tóm tắt bài báo",
    description="Dán link bài báo và nhận tóm tắt tự động bằng mô hình BARTPho."
)

# 5. Chạy demo
demo.launch(share=True)  # share=True để tạo link public


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://697ebeb241302664fd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import gradio as gr
import requests
from bs4 import BeautifulSoup
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# --- Load model và tokenizer ---
MODEL_PATH = "/content/drive/MyDrive/bartpho-summary-model"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)

# --- Hàm sinh tóm tắt ---
def summarize(text, max_input_len=512, max_output_len=128):
    inputs = tokenizer(
        text,
        max_length=max_input_len,
        truncation=True,
        return_tensors="pt"
    )
    summary_ids = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=max_output_len,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True
    )
    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# --- Hàm lấy nội dung từ link báo ---
def get_article_content(url):
    try:
        r = requests.get(url, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")

        # Lấy text trong các thẻ <p>
        paragraphs = soup.find_all("p")
        text = " ".join([p.get_text() for p in paragraphs])

        # Giới hạn độ dài để tránh quá tải
        return text[:3000]
    except:
        return ""

# --- Hàm cho Gradio ---
def summarize_input(url, text):
    if url.strip():  # Nếu có link
        content = get_article_content(url)
        if not content:
            return "❌ Không lấy được nội dung từ link."
        return summarize(content)
    elif text.strip():  # Nếu có văn bản nhập tay
        return summarize(text)
    else:
        return "⚠️ Vui lòng nhập link hoặc văn bản."

# --- Giao diện Gradio ---
with gr.Blocks() as demo:
    gr.Markdown("## 📰 Tóm tắt văn bản/Bài báo bằng BARTPho")

    with gr.Row():
        url_in = gr.Textbox(label="Nhập link bài báo", placeholder="https://...")
    with gr.Row():
        text_in = gr.Textbox(label="Hoặc nhập trực tiếp văn bản", lines=8)
    with gr.Row():
        btn = gr.Button("Tóm tắt")
    output = gr.Textbox(label="Kết quả tóm tắt")

    btn.click(fn=summarize_input, inputs=[url_in, text_in], outputs=output)

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://23073c118e0d49b907.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [3]:
!pip install evaluate bert-score rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.2 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=535ed206feef67cef4d8baccb98aa005786f0778658680bf058abb1030c4f282
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import evaluate

# 1. Load dataset gốc
dataset_path = "/content/drive/MyDrive/dataset_all NLP.csv"  # thay đường dẫn
df = pd.read_csv(dataset_path)   # dataset cần có cột "article" và "summary"

# 2. Chia thành train/val/test
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)   # 80% train, 20% còn lại
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)  # 10% val, 10% test

print("Số mẫu train:", len(train_df))
print("Số mẫu val:", len(val_df))
print("Số mẫu test:", len(test_df))

# 3. Load model đã lưu
model_path = "/content/drive/MyDrive/bartpho_summary_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to("cuda")

# 4. Sinh tóm tắt cho tập test
articles = test_df["article"].tolist()
references = test_df["summary"].tolist()
predictions = []

for text in articles:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to("cuda")
    outputs = model.generate(**inputs, max_length=128, num_beams=4)
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    predictions.append(summary)

# 5. Tính ROUGE
rouge = evaluate.load("rouge")
rouge_results = rouge.compute(predictions=predictions, references=references)
print("ROUGE:", rouge_results)

# 6. Tính BERTScore
bertscore = evaluate.load("bertscore")
bertscore_results = bertscore.compute(predictions=predictions, references=references, lang="vi")
print("BERTScore F1 trung bình:", sum(bertscore_results["f1"]) / len(bertscore_results["f1"]))


Số mẫu train: 3041
Số mẫu val: 380
Số mẫu test: 381


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


ROUGE: {'rouge1': np.float64(0.9494490129078954), 'rouge2': np.float64(0.9117830826584641), 'rougeL': np.float64(0.9248661640190629), 'rougeLsum': np.float64(0.9241223083746956)}


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

BERTScore F1 trung bình: 0.9695167552454891


In [5]:
test_df.to_csv("/content/drive/MyDrive/test_set.csv", index=False, encoding="utf-8-sig")
print("Đã lưu tập test vào /content/drive/MyDrive/test_set.csv")


Đã lưu tập test vào /content/drive/MyDrive/test_set.csv
